# L01 -- Power Flow Fundamentals

Companion notebook to the L01 lecture. After running this you should be able to:
- Build a 3-bus per-unit system in pandapower
- Reproduce the worked-example voltages from the slides
- Cross-check Newton-Raphson against the backward/forward sweep solver
- Read `net.res_bus` and `net.res_line` for the quantities the project cares about

**Prerequisite**: micromamba env `pdpower` is active. Slides: `L01_power_flow_fundamentals.pdf`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import pandapower as pp
from packaging.version import Version
import matplotlib.pyplot as plt

print(f"pandapower {pp.__version__}, numpy {np.__version__}, pandas {pd.__version__}")
assert Version(pp.__version__) >= Version("3.2.1"), "Need pandapower >= 3.2.1 (env: pdpower)"

## 1. Build the 3-bus worked-example system

Slack at bus 1 ($V = 1.00 \angle 0$), PQ loads at buses 2 and 3.

On a 100 MVA / 20 kV base, the per-unit values from the slide map to:
- Line 1--2: $z = 0.02 + j0.06$ pu  $\;\Rightarrow$ 0.08 + j0.24 ohm
- Line 2--3: $z = 0.03 + j0.08$ pu  $\;\Rightarrow$ 0.12 + j0.32 ohm
- Load at bus 2: $S_2 = 0.45 + j0.18$ pu = 45 MW + 18 MVAR
- Load at bus 3: $S_3 = 0.30 + j0.12$ pu = 30 MW + 12 MVAR

In [ ]:
def build_l01_system():
    net = pp.create_empty_network(sn_mva=100.0)
    b1 = pp.create_bus(net, vn_kv=20.0, name="slack")
    b2 = pp.create_bus(net, vn_kv=20.0, name="bus2")
    b3 = pp.create_bus(net, vn_kv=20.0, name="bus3")
    pp.create_ext_grid(net, bus=b1, vm_pu=1.0)
    pp.create_line_from_parameters(net, b1, b2, length_km=1.0,
        r_ohm_per_km=0.08, x_ohm_per_km=0.24, c_nf_per_km=0.0, max_i_ka=10.0)
    pp.create_line_from_parameters(net, b2, b3, length_km=1.0,
        r_ohm_per_km=0.12, x_ohm_per_km=0.32, c_nf_per_km=0.0, max_i_ka=10.0)
    pp.create_load(net, bus=b2, p_mw=45.0, q_mvar=18.0)
    pp.create_load(net, bus=b3, p_mw=30.0, q_mvar=12.0)
    return net

net = build_l01_system()
print(net.bus)
print(net.line)
print(net.load)

## 2. Solve with Newton-Raphson and read results

`pp.runpp(net)` runs Newton-Raphson by default. After it returns we expect:
- `net.converged == True`
- `vm_pu[bus2] ≈ 0.964`, `va_degree[bus2] ≈ -2.3`
- `vm_pu[bus3] ≈ 0.944`, `va_degree[bus3] ≈ -3.6`

In [ ]:
pp.runpp(net)
print("converged:", net.converged)
print(net.res_bus.round(4))
print(net.res_line.round(4))

In [ ]:
# Verification against slide values (tolerance 1e-3 pu)
assert net.converged, "Power flow did not converge"
assert abs(net.res_bus.vm_pu[1] - 0.964) < 1e-3, f"V_2 mismatch: {net.res_bus.vm_pu[1]:.4f}"
assert abs(net.res_bus.vm_pu[2] - 0.944) < 1e-3, f"V_3 mismatch: {net.res_bus.vm_pu[2]:.4f}"
assert abs(net.res_bus.va_degree[1] - (-2.3)) < 0.1
assert abs(net.res_bus.va_degree[2] - (-3.6)) < 0.1
print("All L01 worked-example assertions PASS.")

## 3. Cross-check: backward/forward sweep solver

For a radial network, BFSW should give the same result as NR (to numerical precision).

In [ ]:
net_bfsw = build_l01_system()
pp.runpp(net_bfsw, algorithm="bfsw")
print(net_bfsw.res_bus[["vm_pu", "va_degree"]].round(6))

# Should match NR result very closely
diff = (net.res_bus.vm_pu - net_bfsw.res_bus.vm_pu).abs().max()
print(f"max |V_NR - V_BFSW| = {diff:.2e}")
assert diff < 1e-4, "NR and BFSW disagree -- inspect the network setup"

## 4. Visualize the voltage profile

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot([1, 2, 3], net.res_bus.vm_pu, "o-", color="C0", label="$|V|$ (pu)")
ax.axhline(0.95, color="red", linestyle="--", alpha=0.6, label="0.95 limit")
ax.axhline(1.05, color="red", linestyle="--", alpha=0.6)
ax.set_xlabel("bus index"); ax.set_ylabel("$|V|$ (pu)")
ax.set_xticks([1, 2, 3]); ax.set_ylim(0.90, 1.05); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Homework

### Required
1. Double both line impedances (multiply `r_ohm_per_km` and `x_ohm_per_km` by 2). Re-run NR. Which bus moves the most, and why?
2. Add a PV-type generator at bus 3 with $|V|=1.02$. Use `pp.create_gen(net, bus=b3, p_mw=10, vm_pu=1.02)`. Re-run. How do the angles change?
3. Drop the load at bus 3 to zero. Predict the new voltages by hand using the linearized formula $\Delta V \approx (RP+XQ)/V$ before running.

### Optional
- Implement a single Newton-Raphson iteration in pure numpy starting from flat start. Compare your $\Delta\theta$, $\Delta V$ to pandapower's internal trace.

In [ ]:
# TODO Q1: doubled impedances
# ...

# TODO Q2: PV-type generator at bus 3
# ...

# TODO Q3: zero load at bus 3, predicted vs solved voltages
# ...